# GAMPixPy Batch Simulation

This notebook is meant to show the workflow for processing many events in series with the GAMPixPy framework.  It follows a very similar script which you can find in `examples/batch_sim.py`

First, import some necessary modules from the gampixpy library

In [1]:
from gampixpy import detector, input_parsing, plotting, config, output

Next, some related libraries which are needed for this notebook:

In [2]:
# a fancy progress bar
import tqdm

# import torch and set the default device (needed for GPU)
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    # Set the default device to CUDA
    torch.set_default_device(device)
    print(f"Default device set to: {torch.cuda.get_device_name(device)}")
else:
    device = torch.device('cpu')
    print("CUDA is not available, using CPU")

CUDA is not available, using CPU


## Configuration

The first step in simulation is to define our detector and how it will behave.  These configuration parameters are divided into three categories:

- *detector config*: defines the physical size and location of the detector volume(s).
- *physics config*: defines the physical properties of the detector material (LAr) and parameters pertaining to drift mechanics and charge yield.  These parameters very rarely need to be changed, and the default is usually fine.
- *readout config*: defines the details of the anode plane e.g., pitches for coarse tiles and pixels, noise levels, thresholds, etc.

These config objects act like dictionaries.  They will be used at various points by the library, so keep them handy!

After defining these, we'll use them to create a detector model.

In [3]:
conf = config.ConfigManager(detector_config='far_detector_vd',
                            readout_config='demo_large_pixels')

## We can also build one of these from a yaml file
## The configs which come along with the library are
## accessible from the `preset_[...]_configs`, but can
## also be found alongside the library code
# readout_config = config.ReadoutConfig('../../gampixpy/readout_config/GAMPixD_notruth.yaml')

detector_model = detector.DetectorModel(config_manager = conf)

## Input Parsing
Next, we need to read some events from a file on disk.  Included with this library are a few example inputs.  We'll read some edep-sim output which has been dumped to HDF5 format, but we can also read directly from a ROOT file in the ROOTracker format

In [4]:
input_file = '../inputs/muon_1-4GeV.h5'
input_parser = input_parsing.EdepSimParser(input_file,
                                           config_manager = conf)

## File Output

To record simulation outputs, we'll initialize an output manager before we start iterating.  This allows us to write each event to disk as we go, so that if something goes wrong, we still have a partial record of the output.

In [5]:
!rm test_output.h5
output_manager = output.OutputManager('test_output.h5',
                                     config_manager = conf)

## Simulation

To process events in serial, we use the `InputParser`'s iterator interface:

In [6]:
for event_index, event_data, event_meta in tqdm.tqdm(input_parser):
    # run the `simulate` method with diagnostic information suppressed
    detector_model.simulate(event_data, verbose = False)

    output_manager.add_entry(event_data,
                             event_meta,
                             event_id = event_index)
    # by default, the output will create a new id which is just indexed from 0
    # here, we specify the output id is the same as the event index from the input
    # this matters when there are "empty" events, which will produce no output

100%|██████████████████████████████████████████████████████████| 10/10 [02:34<00:00, 15.50s/it]
